# Sprint 5

## Install PySpark

In [15]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [16]:
# Windows warning

In [17]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 4.1.1
Shuffle partitions: 8


## Import the funtions and create data path

In [18]:
from pathlib import Path
from urllib.request import urlretrieve # to download data if not already present

from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    broadcast
)

# Path for MIMIC-IV data
DATA_DIR = Path("../data/MIMIC-IV/hosp")


### Prompting file locations

In [19]:
# --- Prompt for data directory locations ---
def prompt_dir(prompt_text):
    val = input(f"{prompt_text}: ").strip()
    return Path(val).resolve() # resolve() turns ../data/MIMIC-IV/hosp into the full absolute path automatically


# Prompt user for each data location
DATA_DIR = prompt_dir("Path to MIMIC hosp dataset")
GENERAL_DATA_DIR = prompt_dir("Path to general data directory")
EVIDENCE_DIR = prompt_dir("Path to out/evidence directory")

# Required files from the MIMIC hospital dataset
hosp_files = ["d_icd_diagnoses.csv.gz", "admissions.csv.gz", "diagnoses_icd.csv.gz", "patients.csv.gz"]
# Required files from the general data directory
general_files = ["bc_icd_codes.csv", "symptom_icd_list.txt"]
# Required files from out/evidence
evidence_files = ["pre_bc_symptom_timeline.tsv"]

print("\nChecking MIMIC hosp files in:", DATA_DIR)
for f in hosp_files:
    p = DATA_DIR / f
    print(f"  {'OK' if p.exists() else 'ERROR: missing'}  {f}")

print("\nChecking general data files in:", GENERAL_DATA_DIR)
for f in general_files:
    p = GENERAL_DATA_DIR / f
    print(f"  {'OK' if p.exists() else 'ERROR: missing'}  {f}")

print("\nChecking out/evidence files in:", EVIDENCE_DIR)
for f in evidence_files:
    p = EVIDENCE_DIR / f
    print(f"  {'OK' if p.exists() else 'ERROR: missing'}  {f}")



Checking MIMIC hosp files in: /Users/kristychan/Desktop/bladdards-MIMIC-health/data/MIMIC-IV/hosp
  OK  d_icd_diagnoses.csv.gz
  OK  admissions.csv.gz
  OK  diagnoses_icd.csv.gz
  OK  patients.csv.gz

Checking general data files in: /Users/kristychan/Desktop/bladdards-MIMIC-health/data
  OK  bc_icd_codes.csv
  OK  symptom_icd_list.txt

Checking out/evidence files in: /Users/kristychan/Desktop/bladdards-MIMIC-health/out/evidence
  OK  pre_bc_symptom_timeline.tsv


## Dataframes

In [20]:
# -------------------------------------------------------
# Visits dataframe
# -------------------------------------------------------


In [21]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION — diagnoses DataFrame
# Collects all diagnoses for visits of interest
# (pre-BC symptom visits + first BC diagnosis visits)
# -------------------------------------------------------

EVIDENCE_DIR = Path("../out/evidence")
# DATA_DIR is already defined above as Path("data/MIMIC-IV/hosp")

# STEP 1: Load visits of interest from pre_bc_symptom_timeline
# row_type (SYMPTOM or BC_FIRST_DX) becomes visit_type
timeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(EVIDENCE_DIR / "pre_bc_symptom_timeline.csv"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("row_type").alias("visit_type")
    )
    .dropDuplicates(["hadm_id"])  # one visit_type label per admission
)

print("Timeline visits of interest:", timeline_df.count())
timeline_df.show(5)

# STEP 2: Load all diagnoses from MIMIC
# seq_num = order diagnoses were recorded per visit → becomes ranking
dx_icd_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "diagnoses_icd.csv.gz"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("seq_num").cast("int").alias("ranking"),
        col("icd_code"),
        col("icd_version").cast("int")
    )
)

print("diagnoses_icd rows:", dx_icd_df.count())
dx_icd_df.show(5)

# STEP 3: Load ICD code dictionary
# Maps icd_code + icd_version → human readable description
icd_dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code"),
        col("icd_version").cast("int"),
        col("long_title").alias("icd_desc")
    )
)

print("ICD dictionary rows:", icd_dict_df.count())
icd_dict_df.show(5)

# STEP 4: Filter diagnoses to visits of interest only
# Inner join on hadm_id — keeps only admissions in our timeline
# broadcast(timeline_df) since it is small (1568 rows vs 6M+)
filtered_dx_df = dx_icd_df.join(
    broadcast(timeline_df),
    on="hadm_id",
    how="inner"
)

print("Diagnoses for visits of interest:", filtered_dx_df.count())

# Drop duplicate subject_id introduced by the join
# (both dx_icd_df and timeline_df have subject_id)
filtered_dx_df2 = filtered_dx_df.drop(timeline_df["subject_id"])

# STEP 5: Enrich with ICD descriptions
# Left join on icd_code + icd_version — must match both since
# same code can mean different things in ICD-9 vs ICD-10
diagnoses = (
    filtered_dx_df2.join(
        broadcast(icd_dict_df),  # dictionary is small, broadcast it
        on=["icd_code", "icd_version"],
        how="left"  # keep all rows even if no dictionary entry found
    )
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("visit_type").cast("string"),
        col("ranking").cast("int"),
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("icd_desc").cast("string")
    )
    .orderBy("subject_id", "hadm_id", "ranking")
)

print("=== diagnoses DataFrame ===")
print("Row count:", diagnoses.count())
diagnoses.printSchema()
diagnoses.show(10, truncate=False)


Timeline visits of interest: 1568
+----------+--------+-----------+
|subject_id| hadm_id| visit_type|
+----------+--------+-----------+
|  10383113|20007405|BC_FIRST_DX|
|  13470381|20010741|BC_FIRST_DX|
|  15129856|20010894|BC_FIRST_DX|
|  13349232|20015647|BC_FIRST_DX|
|  15764116|20020797|    SYMPTOM|
+----------+--------+-----------+
only showing top 5 rows


diagnoses_icd rows: 6364488
+----------+--------+-------+--------+-----------+
|subject_id| hadm_id|ranking|icd_code|icd_version|
+----------+--------+-------+--------+-----------+
|  10000032|22595853|      1|    5723|          9|
|  10000032|22595853|      2|   78959|          9|
|  10000032|22595853|      3|    5715|          9|
|  10000032|22595853|      4|   07070|          9|
|  10000032|22595853|      5|     496|          9|
+----------+--------+-------+--------+-----------+
only showing top 5 rows
ICD dictionary rows: 112107
+--------+-----------+--------------------+
|icd_code|icd_version|            icd_desc|
+--------+-----------+--------------------+
|    0010|          9|Cholera due to vi...|
|    0011|          9|Cholera due to vi...|
|    0019|          9|Cholera, unspecified|
|    0020|          9|       Typhoid fever|
|    0021|          9| Paratyphoid fever A|
+--------+-----------+--------------------+
only showing top 5 rows


Diagnoses for visits of interest: 24054
=== diagnoses DataFrame ===


Row count: 24054
root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- visit_type: string (nullable = true)
 |-- ranking: integer (nullable = true)
 |-- icd_code: string (nullable = true)
 |-- icd_version: integer (nullable = true)
 |-- icd_desc: string (nullable = true)



+----------+--------+-----------+-------+--------+-----------+-------------------------------------------+
|subject_id|hadm_id |visit_type |ranking|icd_code|icd_version|icd_desc                                   |
+----------+--------+-----------+-------+--------+-----------+-------------------------------------------+
|10001401  |21544441|BC_FIRST_DX|1      |C675    |10         |Malignant neoplasm of bladder neck         |
|10001401  |21544441|BC_FIRST_DX|2      |I10     |10         |Essential (primary) hypertension           |
|10001401  |21544441|BC_FIRST_DX|3      |D259    |10         |Leiomyoma of uterus, unspecified           |
|10001401  |21544441|BC_FIRST_DX|4      |Z87891  |10         |Personal history of nicotine dependence    |
|10001401  |21544441|BC_FIRST_DX|5      |E785    |10         |Hyperlipidemia, unspecified                |
|10001401  |21544441|BC_FIRST_DX|6      |E890    |10         |Postprocedural hypothyroidism              |
|10015568  |26581506|BC_FIRST_DX|1   

In [22]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION for icd_codes lookup table
# Goal: all ICD codes with a status label
# NOT_RELATED = general code
# RELEVANT = symptom related to BC (from symptom_icd_list.txt)
# BC_DIAGNOSIS = confirmed BC code (from bc_icd_codes.csv)
# -------------------------------------------------------

# Step 1: Load full ICD dictionary as base
icd_codes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("long_title").alias("description")
    )
)

print("Total ICD codes:", icd_codes.count())
icd_codes.show(5)

# Step 2: Load BC diagnosis codes from sprint 2 output
bc_codes_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/bc_icd_codes.csv")
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int")
    )
)

print("BC diagnosis codes:", bc_codes_df.count())
bc_codes_df.show(5)

# Step 3: Load symptom ICD codes from sprint 3 output
symptom_codes_df = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv("../data/symptom_icd_list.txt")
    .toDF("icd_code", "icd_version")
    .filter(col("icd_code") != "icd_code")
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int")
    )
)

print("Symptom codes:", symptom_codes_df.count())
symptom_codes_df.show(5)

# Step 4: Left join BC codes onto full ICD list
# flag = 1 if it's a BC diagnosis code
icd_with_bc = icd_codes.join(
    broadcast(bc_codes_df.withColumn("is_bc", col("icd_code").isNotNull())),
    on=["icd_code", "icd_version"],
    how="left"
)

# Step 5: Left join symptom codes
# flag = 1 if it's a relevant symptom code
icd_with_both = icd_with_bc.join(
    broadcast(symptom_codes_df.withColumn("is_symptom", col("icd_code").isNotNull())),
    on=["icd_code", "icd_version"],
    how="left"
)

# Step 6: Derive status column
# BC_DIAGNOSIS takes priority over RELEVANT
from pyspark.sql.functions import when

icd_codes = (
    icd_with_both
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("description").cast("string"),
        when(col("is_bc") == True, "BC_DIAGNOSIS")
        .when(col("is_symptom") == True, "RELEVANT")
        .otherwise("NOT_RELATED")
        .alias("status")
    )
    .orderBy("icd_code", "icd_version")
)

print("=== icd_codes lookup table ===")
print("Row count:", icd_codes.count())
icd_codes.printSchema()
icd_codes.show(10, truncate=False)

# Quick check — how many of each status?
print("Status distribution:")
icd_codes.groupBy("status").agg(count("*").alias("count")).show()

Total ICD codes: 112107
+--------+-----------+--------------------+
|icd_code|icd_version|         description|
+--------+-----------+--------------------+
|    0010|          9|Cholera due to vi...|
|    0011|          9|Cholera due to vi...|
|    0019|          9|Cholera, unspecified|
|    0020|          9|       Typhoid fever|
|    0021|          9| Paratyphoid fever A|
+--------+-----------+--------------------+
only showing top 5 rows
BC diagnosis codes: 26
+--------+-----------+
|icd_code|icd_version|
+--------+-----------+
|     C67|         10|
|    C670|         10|
|    C671|         10|
|    C672|         10|
|    C673|         10|
+--------+-----------+
only showing top 5 rows
Symptom codes: 162
+--------+-----------+
|icd_code|icd_version|
+--------+-----------+
|   30653|          9|
|   57400|          9|
|   57401|          9|
|   57410|          9|
|   57411|          9|
+--------+-----------+
only showing top 5 rows
=== icd_codes lookup table ===
Row count: 112107
roo

In [23]:
# -------------------------------------------------------
# KRISTY'S SECTION
# 1st BC Diagnoses Rankings
# -------------------------------------------------------

from pyspark.sql.functions import min as spark_min, mean, max as spark_max, approx_percentile

# Filter diagnoses to BC_FIRST_DX visits only
bc_dx_visits = diagnoses.filter(col("visit_type") == "BC_FIRST_DX")

# Keep only rows where the icd_code matches a known BC diagnosis code
# Inner join on icd_code + icd_version against bc_codes_df
bc_dx_relevant = bc_dx_visits.join(
    broadcast(bc_codes_df),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("BC diagnosis rows in BC_FIRST_DX visits:", bc_dx_relevant.count())

# Find the highest-ranked (lowest ranking number) BC diagnosis per subject
# ranking 1 = listed first = most important
bc_top_ranking = (
    bc_dx_relevant
    .groupBy("subject_id")
    .agg(spark_min("ranking").alias("top_ranking"))
)

print("Subjects with a BC diagnosis code:", bc_top_ranking.count())

# Aggregate statistics across all subjects' top rankings
bc_box_plot = (
    bc_top_ranking
    .agg(
        mean("top_ranking").alias("MEAN"),
        spark_min("top_ranking").alias("MIN"),
        spark_max("top_ranking").alias("MAX"),
        approx_percentile("top_ranking", 0.25).alias("Q1"),
        approx_percentile("top_ranking", 0.50).alias("Q2 (Median)"),
        approx_percentile("top_ranking", 0.75).alias("Q3"),
        count("top_ranking").alias("Total Count")
    )
)

print("BC Diagnosis Box Plot")
bc_box_plot.show(truncate=False)


BC diagnosis rows in BC_FIRST_DX visits: 1171


Subjects with a BC diagnosis code: 1143
BC Diagnosis Box Plot


+-----------------+---+---+---+-----------+---+-----------+
|MEAN             |MIN|MAX|Q1 |Q2 (Median)|Q3 |Total Count|
+-----------------+---+---+---+-----------+---+-----------+
|5.619422572178478|1  |35 |1  |4          |8  |1143       |
+-----------------+---+---+---+-----------+---+-----------+



In [24]:
# -------------------------------------------------------
# Aggregation
# -------------------------------------------------------


# -------------------------------------------------------
# BHOOMIKA'S SECTION — Symptom Diagnosis Rankings
# Goal: Check how relevant symptom visits are by finding
# the minimum ranking of relevant ICD codes per visit
# -------------------------------------------------------

from pyspark.sql.functions import min as spark_min, avg as spark_avg, count

# Filter diagnoses to symptom visits only
symptom_visits_df = diagnoses.filter(col("visit_type") == "SYMPTOM")

# Get only RELEVANT codes from our icd_codes lookup table
relevant_icd_df = icd_codes.filter(col("status") == "RELEVANT")

# Inner join — keep only symptom rows where icd_code is RELEVANT
relevant_symptoms_df = symptom_visits_df.join(
    broadcast(relevant_icd_df),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("Relevant symptom rows:", relevant_symptoms_df.count())
relevant_symptoms_df.show(5)

# Group by visit (hadm_id) and find minimum ranking per visit
# Lower ranking = more important diagnosis
symptom_rankings = (
    relevant_symptoms_df
    .groupBy("hadm_id")
    .agg(
        spark_min(col("ranking")).alias("symptom_rankings")
    )
)

print("=== Symptom Rankings Summary ===")
symptom_rankings.select("symptom_rankings").summary().show()


Relevant symptom rows: 547
+--------+-----------+----------+--------+----------+-------+--------------------+--------------------+--------+
|icd_code|icd_version|subject_id| hadm_id|visit_type|ranking|            icd_desc|         description|  status|
+--------+-----------+----------+--------+----------+-------+--------------------+--------------------+--------+
|   R1031|         10|  10120826|27121829|   SYMPTOM|      4|Right lower quadr...|Right lower quadr...|RELEVANT|
|    5990|          9|  10247438|23745352|   SYMPTOM|      2|Urinary tract inf...|Urinary tract inf...|RELEVANT|
|   78820|          9|  10247438|23745352|   SYMPTOM|     12|Retention of urin...|Retention of urin...|RELEVANT|
|    R310|         10|  10247438|29483315|   SYMPTOM|     14|     Gross hematuria|     Gross hematuria|RELEVANT|
|    5990|          9|  10255052|23614192|   SYMPTOM|      2|Urinary tract inf...|Urinary tract inf...|RELEVANT|
+--------+-----------+----------+--------+----------+-------+--------

+-------+-----------------+
|summary| symptom_rankings|
+-------+-----------------+
|  count|              425|
|   mean|5.863529411764706|
| stddev|5.351354242699189|
|    min|                1|
|    25%|                2|
|    50%|                4|
|    75%|                8|
|    max|               29|
+-------+-----------------+



## Clean up
Stop spark session when done

In [25]:
# Uncomment when you are completely done:

# spark.stop()